In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration 

In [3]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data.sample(10)

,id,dialogue,summary
4774,13819808,"Peter: Guys, what about our task for tomorrow?...",Winston will have the data filled by tonight. ...
7113,13829165,Rhona: Hey! How are you? Enjoying the holidays...,Ollie's travelling a lot these holidays. She's...
8514,13680628,"Ned: Hey! Sorry, but today I'm out.\r\nStan: ?...",Stan will miss his workout with Ned today. Sta...
3784,13810731,Kaya: Can i ask your brother to help me to sol...,Kaya can't solve a puzzle and is looking for h...
14319,13862626,Rosalie: Hey Mark 🙂\nRosalie: Can you help me?...,Rosalie is going to buy a new Android phone.
13076,13828834,Mona: Do you know where Maria bought this red ...,Mona and Clair think Maria looked amazing in t...
4956,13717230,Hazel: is anyone going by car?\r\nEvan: don't ...,Rob will pick up Hazel with his car. Evan take...
437,13680460,Helen: hi sarah can i came over today to see t...,Helen will come to Sarah at 2:30 today to see ...
629,13611519,Ralph: You still here someplace?\r\nMary: Bath...,Mary is in the bathroom.
11319,13727671,Rose: Did you get the tv? on black friday?\r\n...,Henry didn't buy the TV on Black Friday.


In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

In [9]:
#Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [10]:
train_data.shape

(4000, 3)

# Data Pre-Processing

In [11]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", "", text) #line
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags
    text = text.strip().lower()
    return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)

In [13]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interestingviolet:  claire: hi! :) thanks, but i've already read it. :)claire: but thanks for thinking about me :)"

# Tokenize

In [14]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [15]:
# Raw data ==> tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token_ids = add to input as labels

    return inputs

In [16]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [17]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 11275, 15, 17, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [18]:
# input_ids - dialogue => converted to tokens (ids)
# in input_ids; 1 => EOS, 0 => padding(because we have defined padding="max_length"; means-512; and padding adds extra 0s at the end of sequence) and if we print len of input_ids = 512
len(train_dataset[0]["input_ids"])

# attention_mask ; this tell us what are valid values in input_ids
#in input_ids => for valid values attention_mask val = 1 and for padding val attention_mask val=0

#labels
# here also 1=> EOS and 0 => padding
len(train_dataset[0]["labels"])

150

In [19]:
type(train_dataset)
type(val_dataset)

list

# Working with our Model

In [20]:
#NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [21]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device = ", device)
model.to(device)

device =  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [22]:
# Training Arguments
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [23]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

# Train the model